## Q2 Data-Driven Discovery of a Discrete-Time Recurrence 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import sklearn
from sklearn.metrics import mean_absolute_error as sklearn_mae
from sklearn.metrics import mean_squared_error as sklearn_mse

### > Include username in plots

In [ ]:
username = "gracy.garg"
def add_username_to_plot(username="gracy.garg"):
    """
    Adds the username to the top-right corner of the current plot.
    """
    ax = plt.gca()
    # For 3D axes, use text2D to place text in axes (fractional) coordinates
    if getattr(ax, "name", "") == "3d" or ax.__class__.__name__ == "Axes3D":
        ax.text2D(0.95, 0.95, username, ha="right", va="top", transform=ax.transAxes,
                  fontsize=10, color="gray", alpha=0.7)
    else:
        plt.text(0.95, 0.95, username, ha="right", va="top",
                 transform=plt.gca().transAxes, fontsize=10, color="gray", alpha=0.7)

In [ ]:
DATA_DIR = '../data/Q2'
OUTPUT_DIR='./output'
DATA_FILE = f'{DATA_DIR}/recurrence_timeseries.csv'
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

## 2.1 Dataset and Problem Setup

In [ ]:
from data_prep import RecurrenceData
from analyze import identify_analytical_recurrence, evaluate_single_step, autoregressive_generation, plot_error_vs_horizon, plot_predictions
from train import train_model, tune_history_length
from parsimony import generate_parsimony_comparison
from model import (
    RNNPredictor,
    LSTMPredictor,
    MLPPredictor,
    CNNPredictor,
    LinearARPredictor
)

## 2.2 Sequence Prediction

In [ ]:
# --- Configuration ---
P_CANDIDATES = [3, 5, 8]     # History lengths to try
FORECAST_HORIZON = 50        
LEARNING_RATE = 0.005
MAX_EPOCHS = 100
PATIENCE = 10
BATCH_SIZE = 32


# Q2.2 Tuning: Find optimal history length (p)
best_p, best_dp, p_results = tune_history_length(
    RecurrenceData, RNNPredictor, P_CANDIDATES, 
    epochs=30, batch_size=BATCH_SIZE, lr=LEARNING_RATE, patience=5
)

plt.figure(figsize=(10, 6))
plt.plot(list(p_results.keys()), list(p_results.values()), marker='o', linestyle='--')
plt.title('Validation Curve: MSE vs. History Length (p)')
plt.xlabel('History Length (p)')
plt.ylabel('Best Validation MSE')
plt.xticks(P_CANDIDATES)
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
add_username_to_plot(username)
plt.savefig(f"{OUTPUT_DIR}/validation_curve_p.png")
plt.show()
print(f"Optimal history length (p): {best_p}")

# Q2.2 Final Training: Load data with optimal p
X_train, y_train, X_val, y_val, X_test, y_test = best_dp.load_and_split()

# 1. Train DL RNN
rnn_model = RNNPredictor(input_size=1, hidden_size=32, num_layers=1, output_size=1)
rnn_model, rnn_best_val_mse = train_model(
    rnn_model, X_train, y_train, X_val, y_val, 
    epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE, patience=PATIENCE
)
print(f"\nFinal DL RNN Val MSE: {rnn_best_val_mse:.6f}")

# 2. Train Analytical (Linear AR) baseline (Note: Not strictly necessary for F_theta, 
# but useful for direct DL vs Linear baseline comparison)
ar_model = LinearARPredictor(history_length=best_p)
ar_model, ar_best_val_mse = train_model(
    ar_model, X_train, y_train, X_val, y_val, 
    epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE, patience=PATIENCE
)
print(f"Final Linear AR Val MSE: {ar_best_val_mse:.6f}")


## 2.3 Analytical Recurrence Identification

In [ ]:
# Q2.3 Identification: Derive F_theta
lin_reg_model, analytical_predictor_func = identify_analytical_recurrence(
    best_dp, X_train, y_train
)

## 2.4 Evaluation 

In [ ]:
# Q2.4 Single-Step Evaluation
rnn_mae, rnn_mse, y_test_unnorm, rnn_pred_unnorm = evaluate_single_step(
    rnn_model, X_test, y_test, best_dp
)
X_test_unnorm = np.zeros_like(X_test.squeeze(-1))
for i in range(best_p):
    X_test_unnorm[:, i] = best_dp.inverse_transform(X_test.squeeze(-1)[:, i].reshape(-1, 1)).flatten()

anal_pred_unnorm = analytical_predictor_func(X_test_unnorm)
# Ensure y_test_unnorm and anal_pred_unnorm are clean, flat, float64 NumPy arrays.
y_true_clean = y_test_unnorm.flatten().astype(np.float64)

# anal_pred_unnorm might be a (N,) array or a (N, 1) array. Ensure it's flattened.
anal_pred_clean = anal_pred_unnorm.flatten().astype(np.float64) 
anal_mae = sklearn_mae(y_true_clean, anal_pred_clean)
anal_mse = sklearn_mse(y_true_clean, anal_pred_clean)

print("\n--- Single-Step Test Set Performance ---")
print("{:<15} {:<10} {:<10}".format('Model', 'MAE', 'MSE'))
print("-" * 35)
print("{:<15} {:<10.6f} {:<10.6f}".format('DL RNN', rnn_mae, rnn_mse))
print("{:<15} {:<10.6f} {:<10.6f}".format('Analytical AR', anal_mae, anal_mse))
plot_predictions(y_test_unnorm, rnn_pred_unnorm, anal_pred_unnorm)

In [ ]:
plot_error_vs_horizon(
    best_dp, rnn_model, analytical_predictor_func, X_test, y_test_unnorm, 
    max_horizon=FORECAST_HORIZON
)
print("\nSaved plot to 'output/error_vs_horizon.png'")

# 2.5 Parsimony and Stability Report 

In [ ]:
generate_parsimony_comparison(
    rnn_model, 
    lin_reg_model, 
    best_p, 
    rnn_mae, 
    anal_mae, 
    rnn_best_val_mse, 
    ar_best_val_mse
)

## Short Discussion: History Length & Model Complexity

History Length (p).
The validation curve shows that performance improves as (p) increases, up to the optimal best_p, after which MSE stops decreasing or begins rising.
This illustrates the bias–variance tradeoff:

* Small (p) → insufficient past information → underfitting
* Large (p) → unnecessary noise inclusion → overfitting
* Optimal (p) → minimal validation MSE, best predictive performance

Model Complexity.

* Linear AR:
  Simplest model, very few parameters. Works well if recurrence is nearly linear.
  It serves as a strong analytical baseline but cannot model nonlinearities.

* RNN:
  Best at capturing true recurrence structure because they maintain hidden state.
  LSTM is slightly more stable than RNN due to gating, especially on longer sequences.

* 1D CNN:
  Captures local patterns very efficiently; relatively low parameter count but still nonlinear.
  Performs well when recurrence has short-range dependencies.

Conclusion:
The simplest accurate model depends on the dataset.
